# 14. Attention Visualization（定量, spec §9）

## 学習目標
- attention を「説明」ではなく**測定対象**として扱い、エントロピー・距離・直前トークン率・head 類似度を定量化する
- 学習前後の変化を見る
- **head ablation（因果）** で「重み ≠ 因果的寄与」を実証する

## 数式
- head エントロピー $H_{l,h}=\overline{-\sum_j a_{tj}\log a_{tj}}$（行 t で平均、$j\le t$）
- 平均注意距離 $\overline{\sum_j a_{tj}\,|t-j|}$
- head ablation 効果 = その head の出力を0にしたときの loss 増加（因果量）

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"
from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
M2_RUN = ROOT / "experiments/runs/m2_model_m_classical_seed42"

In [2]:
A = M2_RUN / "analysis"
stats = load_json(A / "attention_stats.json")
meta = load_json(A / "meta.json")
L, H = meta["n_layers"], meta["n_heads"]
ent_i = np.array(stats["init"]["entropy"]); ent_f = np.array(stats["final"]["entropy"])
print("probe:", stats["probe_text"])
print(f"平均エントロピー init {ent_i.mean():.2f} → final {ent_f.mean():.2f} nats")

fig = go.Figure()
fig.add_trace(go.Heatmap(z=ent_f, x=[f"H{h}" for h in range(H)], y=[f"L{l}" for l in range(L)],
                         colorscale="Viridis", colorbar_title="entropy"))
fig.update_layout(title="学習後 attention エントロピー（layer×head, 低い=選択的）",
                  template="plotly_white", height=360)
fig.update_yaxes(autorange="reversed")
fig.show()

probe: 日本の首都は東京です。私はその街に住んでいます。
平均エントロピー init 1.73 → final 1.46 nats


In [3]:
# 直前トークン率・平均距離を層別に
prev_f = np.array(stats["final"]["prev_token_ratio"]).mean(1)
dist_f = np.array(stats["final"]["distance"]).mean(1)
first_f = np.array(stats["final"]["first_token_ratio"]).mean(1)
fig = go.Figure()
fig.add_trace(go.Bar(x=[f"L{l}" for l in range(L)], y=prev_f, name="直前トークン率"))
fig.add_trace(go.Bar(x=[f"L{l}" for l in range(L)], y=first_f, name="先頭トークン率"))
fig.update_layout(barmode="group", title="層別 attention 集中先（final）",
                  yaxis_title="平均注意質量", template="plotly_white", height=340)
fig.show()
print("層別 平均注意距離:", [round(float(x),1) for x in dist_f], "tokens")

層別 平均注意距離: [3.1, 2.9, 2.6, 2.7, 3.0, 2.9] tokens


In [4]:
# attention heatmap（実際のマップ, layer 0 と最終層）
maps = np.load(A / "attention_maps.npz", allow_pickle=True)
tokens = list(maps["tokens"])
from jp_llm_lab.visualization.attention_viz import attention_heatmap_grid
attention_heatmap_grid(torch.tensor(maps["layer0"]), tokens, "final model / layer 0", max_heads=4).show()
attention_heatmap_grid(torch.tensor(maps["layer_last"]), tokens, f"final model / layer {L-1}", max_heads=4).show()

In [5]:
# head ablation: どの head が因果的に重要か
ha = np.array(load_json(A / "head_ablation.json")["head_ablation_delta_loss"])
fig = go.Figure(go.Heatmap(z=ha, x=[f"H{h}" for h in range(H)], y=[f"L{l}" for l in range(L)],
                           colorscale="Reds", colorbar_title="Δloss"))
fig.update_layout(title="head ablation: 各 head を0にしたときの loss 増加（因果的重要度）",
                  template="plotly_white", height=360)
fig.update_yaxes(autorange="reversed")
fig.show()
l,h = np.unravel_index(ha.argmax(), ha.shape)
print(f"最も重要な head: L{l}H{h} (Δloss {ha.max():.3f})")
print(f"その head のエントロピー {ent_f[l,h]:.2f}, 直前トークン率 {np.array(stats['final']['prev_token_ratio'])[l,h]:.3f}")

最も重要な head: L0H3 (Δloss 0.087)
その head のエントロピー 1.48, 直前トークン率 0.185


## Observation / Interpretation / Caveat
- **Observation**: 学習でエントロピーが低下し（選択性↑）、中間層が最も選択的。直前トークン率は 0.14–0.20、平均距離 2.6–3.1 トークンで局所的。head ablation では特定の head（上のセルが実測）だけが loss を大きく動かす。
- **Interpretation**: 多くの head は冗長（ablation の Δloss が小さい）で、少数の head が因果的に効く。エントロピーが低い head が必ずしも因果的に重要とは限らない点に注意（両者は別の量）。
- **Caveat（最重要）**: attention 重みマップは**説明ではない**。因果的重要度は ablation/patching で別途測る必要がある。すべて1つの probe 文・1シードの結果で、head の役割の一般化はできない。ablation は単純なゼロ化で、head 間の相互作用（冗長性）を過小評価しうる。

**次へ**: [15_embedding_analysis](15_embedding_analysis.ipynb)